# 实验一：静态页面与 API 数据采集

课程站点：`https://115.159.159.82/`

依次运行全部单元格，最后会在当前目录生成 `data/devices.json` 和 `data/companies.json`。

In [2]:
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin
import json

import requests
from bs4 import BeautifulSoup
from urllib3.exceptions import InsecureRequestWarning

# 课程站点使用实验证书，仅关闭本实验环境中的证书告警。
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

BASE_URL = "https://115.159.159.82"
session = requests.Session()
session.verify = False

## 任务一：发现 Device API 并观察第一页

In [3]:
api_url = "https://115.159.159.82/api/devices"

r = session.get(
    api_url,
    params={"page": 1, "page_size": 50},
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

data = r.json()
print("keys:", data.keys())
print("page:", data["page"])
print("page_size:", data["page_size"])
print("total:", data["total"])
print("items on this page:", len(data["items"]))
print(data["items"][0])

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json
keys: dict_keys(['page', 'page_size', 'total', 'items'])
page: 1
page_size: 50
total: 5000
items on this page: 50
{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


## 任务二：采集全部 Device

In [5]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

devices = []

for page in range(1, total_pages + 1):
    if page == 1:
        page_data = data
    else:
        r = session.get(
            api_url,
            params={"page": page, "page_size": page_size},
            timeout=10
        )
        r.raise_for_status()
        page_data = r.json()

    devices.extend(page_data["items"])

print("device count:", len(devices))
print("unique device_id:", len({item["device_id"] for item in devices}))

total: 5000
page_size: 50
total_pages: 100
device count: 5000
unique device_id: 5000


## 任务三：解析 Company 静态 HTML

In [6]:
current_url = BASE_URL + "/companies"
r = session.get(current_url, timeout=10)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")
cards = soup.select("article.company-card")

print("本页企业数:", len(cards))

card = cards[0]
company_id = card.get("data-company-id")
name = card.select_one("h2").get_text(strip=True)
status = card.select_one(".badge").get_text(strip=True)

details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print({**details, "company_id": company_id, "name": name, "status": status})

本页企业数: 20
{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台', 'company_id': 'C000001', 'name': '临港精工001有限公司', 'status': '运营中'}


## 任务四：根据“下一页”连续采集 3 页 Company

In [7]:
companies = []
page_counts = []
current_url = BASE_URL + "/companies"

for page_number in range(1, 4):
    r = session.get(current_url, timeout=10)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    cards = soup.select("article.company-card")
    page_companies = []

    for card in cards:
        details = {}
        for item in card.select("dl > div"):
            key = item.select_one("dt").get_text(strip=True)
            value = item.select_one("dd").get_text(strip=True)
            details[key] = value

        page_companies.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2").get_text(strip=True),
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].removesuffix(" 台")),
            "status": card.select_one(".badge").get_text(strip=True)
        })

    companies.extend(page_companies)
    page_counts.append(len(page_companies))
    print(f"第 {page_number} 页企业数:", len(page_companies))

    next_link = soup.select_one("nav.pagination a:last-child")
    if next_link is None:
        break

    current_url = urljoin(current_url, next_link.get("href"))

print("三页企业数:", page_counts)
print("company count:", len(companies))
print("unique company_id:", len({item["company_id"] for item in companies}))

第 1 页企业数: 20
第 2 页企业数: 20
第 3 页企业数: 20
三页企业数: [20, 20, 20]
company count: 60
unique company_id: 60


## 任务五：结果验证（用于截图 4）

In [8]:
device_counts = Counter(item["company_id"] for item in devices)

mismatch_count = 0
for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)

    if expected != actual:
        mismatch_count += 1

print("Company 总数:", len(companies))
print("唯一 company_id 数:", len({item["company_id"] for item in companies}))
print("Device 总数:", len(devices))
print("唯一 device_id 数:", len({item["device_id"] for item in devices}))
print("Company–Device 不一致企业数:", mismatch_count)

Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company–Device 不一致企业数: 0


## 任务六：保存结果

In [9]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录
